# Обучение модели Gemma 4 с помощью Unsloth

Для запуска нажмите "*Runtime*" (Среда выполнения) -> "*Run all*" (Запустить все) на бесплатном инстансе Google Colab с GPU Tesla T4.

Этот ноутбук показывает, как подготовить данные, обучить модель, запустить генерацию и сохранить результат.

Лицензия: [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)

### Новости Unsloth

- **Unsloth Studio**: новый веб-интерфейс без кода для обучения и запуска LLM.
- **Ускорение MoE**: DeepSeek, GLM, Qwen и другие модели обучаются в 12 раз быстрее с экономией 35% видеопамяти.
- **Длинный контекст**: Поддержка ультра-длинного контекста в обучении с подкреплением (RL).

Посетите документацию для полного списка моделей и ноутбуков.

### Установка библиотек

In [ ]:
%%capture
import os, re
# Проверка среды: если это не Colab, устанавливаем базовый unsloth
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Для Colab подбираем совместимые версии xformers под версию torch
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

# Обновляем transformers до нужной версии
!pip install --no-deps transformers==5.5.0
!pip install torchcodec

# Настройка динамической компиляции PyTorch
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
%%capture
!pip install --no-deps --upgrade timm # Необходимо для работы зрения/аудио в Gemma 4

### Загрузка модели Unsloth

In [ ]:
from unsloth import FastVisionModel # Используем FastVisionModel для мультимодальных моделей
import torch

# Список доступных моделей Gemma 4
gemma4_models = [
    # Инструктивные модели (Instruct):
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Базовые модели (Base):
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
]

# Загрузка модели
# load_in_4bit=False использует 16-bit LoRA для лучшего качества
# use_gradient_checkpointing="unsloth" экономит память для длинного контекста
model, processor = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-E2B-it",
    load_in_4bit = False,
    use_gradient_checkpointing = "unsloth",
)

### Настройка LoRA адаптеров

Мы добавляем LoRA адаптеры для эффективной тонкой настройки (fine-tuning). Это позволяет обучать только ~1% параметров модели.

Мы можем настроить, какие части модели обучать:
- vision_layers: визуальные слои (для обработки изображений)
- language_layers: языковые слои (для текста)
- attention_modules / mlp_modules: конкретные блоки внутри слоев

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # Обучать ли визуальные слои
    finetune_language_layers   = True, # Обучать ли языковые слои
    finetune_attention_modules = True, # Обучать ли модули внимания
    finetune_mlp_modules       = True, # Обучать ли MLP слои

    r = 32,                           # Ранг матрицы: выше = точнее, но риск переобучения
    lora_alpha = 32,                  # Масштабирование LoRA (обычно равно r)
    lora_dropout = 0,                 # Dropout для LoRA
    bias = "none",
    random_state = 3407,
    use_rslora = False,               # Rank Stabilized LoRA
    loftq_config = None,              # LoftQ квантование
    target_modules = "all-linear",    # Целевые модули для замены на LoRA
)

### Подготовка данных

В этом примере мы используем датасет с рукописными математическими формулами. 
Задача: преобразовать изображение формулы в формат LaTeX.

Датасет: `unsloth/LaTeX_OCR` (выборка) или `linxy/LaTeX_OCR` (полный).

In [ ]:
from datasets import load_dataset

# Загрузка датасета
dataset = load_dataset("unsloth/LaTeX_OCR", split="train")

# Пример структуры данных:
# dataset[0] содержит ключи 'image' и 'latex'
print(dataset[0])

### Форматирование промптов

Необходимо привести данные к виду, понятному модели. Для Gemma 4 используется специальный шаблон чата.

In [ ]:
def formatting_prompts_func(examples):
    images = examples["image"]
    texts = examples["latex"]
    messages = []
    for image, text in zip(images, texts):
        # Формируем сообщение для мультимодальной модели
        message = [
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "Конвертируй это изображение в LaTeX."}]},
            {"role": "assistant", "content": [{"type": "text", "text": text}]}
        ]
        messages.append(message)
    return { "messages": messages }

# Применяем форматирование
dataset = dataset.map(formatting_prompts_func, batched=True, remove_columns=dataset.column_names)

### Настройка обучения (SFT Trainer)

Здесь мы задаем гиперпараметры: количество эпох, размер батча, скорость обучения.

In [ ]:
from trl import SFTConfig
from unsloth import is_bfloat16_supported

trainer = FastVisionModel.get_trainer(
    model = model,
    processor = processor,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,       # Количество шагов обучения (увеличь для лучшего результата)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Отключаем логирование в wandb для простоты
    ),
)

### Запуск обучения

In [ ]:
# Запуск процесса обучения
trainer.train()

### Сохранение модели

Сохраняем обученные веса LoRA адаптера.

In [ ]:
model.save_pretrained("gemma4_lora_model")
processor.save_pretrained("gemma4_lora_model")
print("Модель сохранена в папку gemma4_lora_model")